# Stream B - Contamination Experiments (Phase 1)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/NLP_project/src')

from data import load_data, make_mixed_dataset
from train import run_finetune
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import gc
import os

In [3]:
# Base settings
model_name = 'gpt2'
total_size = 50000  # Size of fine-tuning dataset
doses = [0.0, 0.01, 0.05, 0.10, 0.25]
seeds = [0, 1, 2]

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
_, _, tweeteval_off = load_data()

def run_contamination(dose, seed):
    print(f"\n=== Running Contamination | Dose: {dose} | Seed: {seed} ===")

    model = AutoModelForCausalLM.from_pretrained(model_name)
    if torch.cuda.is_available():
        model = model.to('cuda')

    mixed_dataset = make_mixed_dataset(tokenizer, dose=dose, seed=seed, total_size=total_size)

    # Split 5% off for eval_loss calculation (perplexity proxy)
    split = mixed_dataset.train_test_split(test_size=0.05, seed=seed)
    train_dataset = split['train']
    eval_dataset = split['test']

    output_dir = f'/content/drive/MyDrive/NLP_project/checkpoints/contaminated/dose_{dose}_seed_{seed}'
    log_csv_path = f'/content/drive/MyDrive/NLP_project/results/contamination_logs/dose_{dose}_seed_{seed}.csv'

    run_finetune(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        output_dir=output_dir,
        eval_dataset=eval_dataset,
        tweeteval_off=tweeteval_off,
        log_csv_path=log_csv_path,
        num_train_epochs=1,
        batch_size=8,
        logging_steps=50
    )

    del model
    gc.collect()
    torch.cuda.empty_cache()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cardiffnlp/tweet_eval' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cardiffnlp/tweet_eval' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Salesforce/wikitext' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Salesforce/wikitext' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cardiffnlp/tweet_eval' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cardiffnlp/tweet_eval' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


In [ ]:
# Execute runs
# You can run a subset of these by changing the lists above, e.g., to run 1 config per Colab session.
for dose in doses:
    for seed in seeds:
        run_contamination(dose, seed)

Output hidden; open in https://colab.research.google.com to view.